In [5]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 定义数据预处理
# ToTensor 将图像转换为 [0.0, 1.0] 的张量
# Normalize 可选，通常用于加速训练收敛
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


In [6]:
# 3. 载入训练集 (包含 50000 张图)
# train=True: torchvision 会自动找到并合并 data_batch_1 至 data_batch_5
ROOT_DIR = 'C:/Jupyter(Anaconda)/data'
CIFAR_dataset = torchvision.datasets.CIFAR10(
    root=ROOT_DIR,
    train=True,
    download=False,
    transform=transform
)

# --- 验证加载情况 ---
print(f"训练集大小: {len(CIFAR_dataset)}")

# 获取单个样本，验证是否返回 Tensor
image, label = CIFAR_dataset[0]
print(f"单图 Shape: {image.shape}, 标签: {label}")



训练集大小: 50000
单图 Shape: torch.Size([3, 32, 32]), 标签: 6


In [7]:
from torch.utils.data import Subset
# 直接切片（仅保留前 N 个，速度最快。⚠️需确保原数据已打乱顺序）
N = 15000  # 修改为你想要的训练样本数量
torch.manual_seed(42)#可复现
idx = torch.randperm(len(CIFAR_dataset))  # 生成随机排列索引

# 1. 包装为子集
train_subset = Subset(CIFAR_dataset, idx[:N])


X_train = torch.stack([img for img, _ in train_subset])
y_train = torch.tensor([lbl for _, lbl in train_subset])

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: torch.Size([15000, 3, 32, 32])
y_train shape: torch.Size([15000])


In [8]:
test_subset=Subset(CIFAR_dataset, idx[N:])
X_test=torch.stack([img for img, _ in test_subset])
y_test=torch.tensor([lbl for _, lbl in test_subset])

In [9]:
X_train = X_train.view(X_train.size(0), -1).float()
X_test = X_test.view(X_test.size(0), -1).float()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: torch.Size([15000, 3072])
X_test shape: torch.Size([35000, 3072])


In [10]:
class NearestNeighbor(object):
  def __init__(self,k=3):
    self.k=k

  def train(self, X, y):
    """ X is N x D where each row is an example. Y is 1-dimension of size N """
    # the nearest neighbor classifier simply remembers all the training data
    self.Xtr = X
    self.ytr = y

  def predict(self, X):
    """ X is N x D where each row is an example we wish to predict label for """
    num_test = X.shape[0]
    # let's make sure that the output type matches the input type
    Ypred = torch.zeros(num_test, dtype = self.ytr.dtype)

    # loop over all test rows
    for i in range(num_test):
      # find the nearest training image to the i'th test image
      # using the L2 distance (ignore sqrt to save a little time)
      distances = torch.sum(torch.square((self.Xtr - X[i,:])), dim=1)
      _, k_indices = torch.topk(distances, self.k, dim=-1, largest=False, sorted=False)
      # ->get the index with k-smallest distance
      k_labels=self.ytr[k_indices]
      Ypred[i] = torch.bincount(k_labels).argmax() # predict the label of the nearest example
    return Ypred

  def predict_vec(self,X):
        # 1. 批量计算 L2 距离矩阵 [num_test, num_train]
        # torch.cdist 默认计算欧氏距离，会自动使用 GPU 并行计算
        distances = torch.cdist(X, self.Xtr)

        # 2. 获取每个测试样本距离最小的 k 个索引 [num_test, k]
        _, k_indices = torch.topk(distances, self.k, dim=-1, largest=False, sorted=False)

        # 3. 提取对应的训练标签 [num_test, k]
        k_labels = self.ytr[k_indices]

        # 4. 沿 k 维度投票，取众数 (mode) 作为预测类别 [num_test]
        Ypred, _ = torch.mode(k_labels, dim=-1)

        return Ypred


In [11]:
nn_5=NearestNeighbor(k=5)
nn_5.train(X_train, y_train)
nn_7=NearestNeighbor(k=7)
nn_7.train(X_train, y_train)

In [12]:
#y_pred=nn_5.predict(X_test)
#acc = (y_pred == y_test).float().mean()
#print ('accuracy: %f' % (acc,))

In [13]:
#y_pred_7=nn_7.predict(X_test)
#acc_ = (y_pred_7 == y_test).float().mean()
#print ('accuracy: %f' % (acc_,))

In [14]:
y_pred=nn_5.predict_vec(X_test)
acc = (y_pred == y_test).float().mean()
print ('accuracy: %f' % (acc,))

accuracy: 0.295543


In [15]:
y_pred_7=nn_7.predict_vec(X_test)
acc_ = (y_pred_7 == y_test).float().mean()
print ('accuracy: %f' % (acc_,))

accuracy: 0.298486


In [17]:
#for i in range(1,200,4):
    #nn_i=NearestNeighbor(k=i)
    #nn_i.train(X_train,y_train)
    #y_pred_i=nn_i.predict_vec(X_test)
    #acc_i = (y_pred_i == y_test).float().mean()
    #print (f'when k={i},accuracy: %f' % (acc_i,))

when k=1,accuracy: 0.299829
when k=5,accuracy: 0.295543
when k=9,accuracy: 0.300943
when k=13,accuracy: 0.302314
when k=17,accuracy: 0.299943
when k=21,accuracy: 0.297200
when k=25,accuracy: 0.295400
when k=29,accuracy: 0.294457
when k=33,accuracy: 0.292886
when k=37,accuracy: 0.291286
when k=41,accuracy: 0.289486
when k=45,accuracy: 0.287114
when k=49,accuracy: 0.287057
when k=53,accuracy: 0.285114
when k=57,accuracy: 0.285143
when k=61,accuracy: 0.283486
when k=65,accuracy: 0.282457
when k=69,accuracy: 0.281057
when k=73,accuracy: 0.279657
when k=77,accuracy: 0.279229
when k=81,accuracy: 0.277886
when k=85,accuracy: 0.276371
when k=97,accuracy: 0.275400
when k=101,accuracy: 0.274400
when k=105,accuracy: 0.274771
when k=109,accuracy: 0.273029
when k=113,accuracy: 0.272200
when k=117,accuracy: 0.271600
when k=121,accuracy: 0.271486
when k=125,accuracy: 0.271314
when k=129,accuracy: 0.270000
when k=133,accuracy: 0.270400
when k=137,accuracy: 0.268171
when k=141,accuracy: 0.267114
when k